# 04 · STAGING: transformar sin ocultar errores

**Objetivo:** explicar normalización, referencias y cálculo correcto de medidas.

**Modalidad:** material para explicar en clase. Los ejemplos se ejecutan únicamente en memoria, con datos ficticios. No conectan a una base, no crean archivos ni ejecutan una ingesta. Los bloques SQL son ilustrativos y no se envían a ningún servidor.

**Prerrequisitos:** Python con pandas. Cada notebook es independiente: no requiere scripts `.py` ni haber ejecutado otra notebook.


In [ ]:
# Importar conexión centralizada desde el módulo
import sys
sys.path.insert(0, '..')

from conexion_sqlserver import obtener_engines, SERVIDOR, BD_ORIGEN, BD_DESTINO, USUARIO

# Obtener los engines
engine_origen, engine_destino = obtener_engines()
print(f'Conexiones establecidas correctamente.')
print(f'Servidor: {SERVIDOR}')
print(f'Origen: {BD_ORIGEN}')
print(f'Destino: {BD_DESTINO}')


## RAW no es el modelo analítico

STAGING toma registros aceptados y aplica reglas explícitas. Cada campo
derivado necesita una definición y un origen. Un INNER JOIN puede ocultar un
huérfano sin avisar: primero debemos diagnosticar y decidir.


In [ ]:
import pandas as pd
from decimal import Decimal

lineas = pd.DataFrame([
    {"linea_id": 1, "factura_id": 100, "producto_id": 10,
     "cantidad": 2, "total": "24.20", "impuesto": "4.20"},
    {"linea_id": 2, "factura_id": 999, "producto_id": 20,
     "cantidad": 1, "total": "6.05", "impuesto": "1.05"},
])
cabeceras = pd.DataFrame([{"factura_id": 100, "cliente_id": 1, "fecha": "2016-01-02"}])
diagnostico = lineas.merge(cabeceras, on="factura_id", how="left",
                          validate="many_to_one", indicator=True)
display(diagnostico)
rechazados = diagnostico.loc[diagnostico["_merge"] != "both"].copy()
aceptados = diagnostico.loc[diagnostico["_merge"] == "both"].copy()
print("Entrada:", len(lineas), "Aceptadas:", len(aceptados), "Rechazadas:", len(rechazados))


In [ ]:
aceptados["total"] = aceptados["total"].map(Decimal)
aceptados["impuesto"] = aceptados["impuesto"].map(Decimal)
aceptados["neto"] = aceptados["total"] - aceptados["impuesto"]
display(aceptados[["linea_id", "cliente_id", "neto", "impuesto", "total"]])


## No confundir total con neto

En WWI `ExtendedPrice` **incluye impuesto**:

`neto = ExtendedPrice - TaxAmount`.

El resultado aceptado es neto **20.00**, impuesto **4.20**, total **24.20**.
No usar float para montos contables. El precio unitario no es una medida
aditiva y un precio NULL no debe imputarse sin una política.

La ciudad obtenida del domicilio actual del cliente no garantiza que fuera la
ciudad histórica de entrega de cada factura.


## Actividad de discusión

¿Qué habría ocurrido con un INNER JOIN? **Respuesta:** la línea de factura 999 desaparecería; el reporte parecería válido pero incompleto.